# Исследование методов обнаружения синтезированной речи в задачах голосовой аутентификации

Инициализация проекта: импорт библиотек, проверка версий, загрузка конфигурации,
настройка зеркала HuggingFace и загрузка датасета с Kaggle.

**Структура проекта:**
- `config.yaml` — все константы проекта в одном месте
- `.env` — секреты (KAGGLE_USERNAME, KAGGLE_KEY, ...), не хранится в git
- `data/raw/` — сырые данные датасета
- `models/`, `logs/` — артефакты обучения


## 1. Базовые библиотеки

In [ ]:
import os
import sys
import json
import csv
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
print("Базовые библиотеки импортированы.")
print(f"Python: {sys.version}")


## 2. Загрузка конфигурации и переменных окружения

`config.yaml` хранит некритичные константы проекта (пути, гиперпараметры и т.д.).
`.env` хранит секреты (Kaggle API ключи и т.п.) и никогда не попадает в git —
см. `.env.example` как шаблон: скопируйте его в `.env` и впишите свои значения.


In [ ]:
import yaml
from dotenv import load_dotenv

# .env должен лежать рядом с этим ноутбуком (см. .env.example)
load_dotenv(dotenv_path=".env")

with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Фиксируем seed из конфига для воспроизводимости
SEED = config["project"]["seed"]
random.seed(SEED)

print(f"Проект: {config['project']['name']}")
print(f"Тема: {config['project']['description']}")
print(f"Seed: {SEED}")


In [ ]:
# Создаём папки проекта, если их ещё нет
for key in ("raw_dir", "processed_dir", "features_dir", "models_dir", "logs_dir", "hf_cache_dir"):
    Path(config["paths"][key]).mkdir(parents=True, exist_ok=True)

print("Структура папок готова:")
for key in ("raw_dir", "processed_dir", "features_dir", "models_dir", "logs_dir", "hf_cache_dir"):
    print(f"  {key}: {config['paths'][key]}")


## 3. Библиотеки для машинного обучения и работы с данными (с проверкой версий)

In [ ]:
import importlib.metadata as importlib_metadata


def check_version(module_name, package_name=None):
    """Импортирует модуль и печатает его версию (или сообщение, что не установлен)."""
    package_name = package_name or module_name
    try:
        module = __import__(module_name)
        try:
            version = importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            version = getattr(module, "__version__", "версия не определена")
        print(f"  [OK]      {module_name:<15} {version}")
        return module
    except ImportError as e:
        print(f"  [ОТСУТСТВУЕТ] {module_name:<15} -> {e}")
        return None


print("Основные ML/DS библиотеки:")
numpy_mod = check_version("numpy")
pandas_mod = check_version("pandas")
matplotlib_mod = check_version("matplotlib")
sklearn_mod = check_version("sklearn", "scikit-learn")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)


## 4. PyTorch и устройство вычислений (CPU/GPU)

In [ ]:
torch_mod = check_version("torch")
torchaudio_mod = check_version("torchaudio")

import torch

device_from_config = config["training"]["device"]
DEVICE = torch.device(device_from_config if torch.cuda.is_available() and device_from_config == "cuda" else "cpu")
print(f"\nИспользуемое устройство: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 5. Специализированные аудио-библиотеки

In [ ]:
print("Аудио-библиотеки:")
librosa_mod = check_version("librosa")
soundfile_mod = check_version("soundfile")

import librosa
import soundfile as sf

SAMPLE_RATE = config["audio"]["sample_rate"]
print(f"\nЦелевая частота дискретизации: {SAMPLE_RATE} Гц")


## 6. HuggingFace Hub: зеркало и transformers

Переменные окружения для зеркала (`HF_ENDPOINT`) выставляются **до** импорта
`transformers`/`huggingface_hub`, чтобы они гарантированно применились.
Если зеркало не нужно — оставьте `mirror_endpoint: ""` в `config.yaml`.


In [ ]:
hf_mirror = config["huggingface"].get("mirror_endpoint", "").strip()
hf_cache_dir = config["huggingface"]["cache_dir"]

if hf_mirror:
    os.environ["HF_ENDPOINT"] = hf_mirror
    print(f"HuggingFace зеркало включено: {hf_mirror}")
else:
    print("HuggingFace зеркало не задано — используется huggingface.co по умолчанию")

os.environ["HF_HOME"] = hf_cache_dir

hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN найден в .env и установлен.")
else:
    print("HF_TOKEN не задан (нужен только для приватных/gated моделей).")


In [ ]:
print("Библиотеки HuggingFace:")
transformers_mod = check_version("transformers")
hf_hub_mod = check_version("huggingface_hub")

import transformers
from huggingface_hub import HfApi

api = HfApi()
print(f"\nHuggingFace endpoint, который будет использоваться: {os.environ.get('HF_ENDPOINT', 'https://huggingface.co (по умолчанию)')}")


## 7. Загрузка датасета с Kaggle

Ключи `KAGGLE_USERNAME` и `KAGGLE_KEY` берутся из `.env` (см. `.env.example`).
Slug датасета задаётся в `config.yaml` (`dataset.kaggle_slug`) — замените на
актуальный, если понадобится переключиться на другой источник данных.


In [ ]:
kaggle_username = os.environ.get("KAGGLE_USERNAME", "")
kaggle_key = os.environ.get("KAGGLE_KEY", "")

if not kaggle_username or not kaggle_key:
    print(
        "KAGGLE_USERNAME / KAGGLE_KEY не найдены в .env.\n"
        "Скопируйте .env.example -> .env и впишите свои данные "
        "(Kaggle -> Account -> Create New API Token)."
    )
else:
    # Переменные окружения должны быть выставлены ДО импорта модуля kaggle
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
    print("Учётные данные Kaggle найдены в .env.")


In [ ]:
if kaggle_username and kaggle_key and config["dataset"].get("download", True):
    from kaggle.api.kaggle_api_extended import KaggleApi

    kaggle_api = KaggleApi()
    kaggle_api.authenticate()

    dataset_slug = config["dataset"]["kaggle_slug"]
    raw_dir = Path(config["paths"]["raw_dir"])

    print(f"Скачивание датасета '{dataset_slug}' в {raw_dir} ...")
    kaggle_api.dataset_download_files(
        dataset_slug,
        path=str(raw_dir),
        unzip=config["dataset"].get("unzip", True),
        quiet=False,
    )
    print("Готово.")
else:
    print("Загрузка датасета пропущена (нет ключей Kaggle или download=false в config.yaml).")


In [ ]:
raw_dir = Path(config["paths"]["raw_dir"])
if raw_dir.exists():
    contents = sorted(raw_dir.iterdir())[:20]
    print(f"Содержимое {raw_dir} (первые 20 элементов):")
    for p in contents:
        print(f"  {'[dir] ' if p.is_dir() else '[file]'} {p.name}")
else:
    print(f"Папка {raw_dir} ещё не создана / пуста.")


## 8. Быстрая проверка на одном аудиофайле

Находит первый `.wav`/`.flac` файл в `raw_dir` и строит его waveform —
удобная проверка, что librosa/soundfile корректно читают данные датасета.


In [ ]:
audio_extensions = (".wav", ".flac", ".mp3", ".ogg")
audio_files = [p for p in raw_dir.rglob("*") if p.suffix.lower() in audio_extensions]

print(f"Найдено аудиофайлов: {len(audio_files)}")

if audio_files:
    sample_path = audio_files[0]
    y, sr = librosa.load(sample_path, sr=SAMPLE_RATE)
    print(f"Пример: {sample_path.name} | длительность: {len(y)/sr:.2f} c | sr={sr}")

    plt.figure()
    librosa.display.waveshow(y, sr=sr)
    plt.title(f"Waveform: {sample_path.name}")
    plt.xlabel("Время, с")
    plt.ylabel("Амплитуда")
    plt.tight_layout()
    plt.show()
else:
    print("Аудиофайлы не найдены — сначала выполните загрузку датасета (шаг 7).")


## Дальнейшие шаги

1. Изучить протоколы датасета (train/dev/eval, разметка bona fide / spoof).
2. Написать `Dataset`/`DataLoader` для PyTorch с извлечением признаков (MFCC / мел-спектрограммы).
3. Собрать baseline (классические признаки + SVM/RandomForest) для сравнения.
4. Обучить нейросетевую модель (CNN на спектрограммах или fine-tune wav2vec2/HuBERT через `transformers`).
5. Оценить по метрикам EER / min-tDCF, зафиксировать результаты.
